# Exercise on Joins and anti-joins: add information from other tables

In [9]:
# import libraries - solution
import pandas as pd
import numpy as np

# Set some Pandas options: maximum number of rows/columns it's going to display
#pd.set_option('display.max_rows', 1000)
#pd.set_option('display.max_columns', 100)

## Load data from clinical trial

Data comes in two different files. The file `predimed_records.csv` file contains the clinical data for each patient. The file 'predimed_location.csv' contain the information about the meaning of the location codes. Load the two dataframes, inspect them and complete the exercise below.

In [4]:
# load the dataframes
# solution
df_patients = pd.read_csv('../../data/predimed_records.csv')
df_patients

,patient-id,location-id,sex,age,smoke,bmi,waist,wth,htn,diab,hyperchol,famhist,hormo,p14,toevent,event
0,436,4,Male,58,Former,33.53,122,0.753086,No,No,Yes,No,No,10,5.374401,Yes
1,1130,4,Male,77,Current,31.05,119,0.730061,Yes,Yes,No,No,No,10,6.097194,No
2,1131,4,Female,72,Former,30.86,106,0.654321,No,Yes,No,Yes,No,8,5.946612,No
3,1132,4,Male,71,Former,27.68,118,0.694118,Yes,No,Yes,No,No,8,2.907598,Yes
4,1111,2,Female,79,Never,35.94,129,0.806250,Yes,No,Yes,No,No,9,4.761123,No
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
6319,120,5,Female,66,Never,28.51,104,0.645963,Yes,No,Yes,Yes,No,8,3.550992,No
6320,118,5,Male,80,Never,23.81,109,0.589189,Yes,Yes,Yes,Yes,No,8,2.743326,No
6321,351,3,Male,57,Former,25.24,100,0.571429,Yes,No,Yes,No,NaN,7,0.479124,No
6322,499,5,Female,71,Never,32.04,98,0.653333,Yes,No,Yes,Yes,No,6,2.587269,No


In [29]:
# solution
df_locations = pd.read_csv('../../data/predimed_location.csv')
df_locations

,Unnamed: 0,location-id,city
0,0,4,Madrid
1,1,1,Valencia
2,2,2,Barcelona
3,3,5,Bilbao
4,4,3,Malaga


There were 5 different locations where the study was conducted, each one gave an identification number `patient-id` to each participant.

In [20]:
# solution
df_locations['location-id'].unique()

array([4, 1, 2, 5, 3])

## Exercise 1: Add location information (the city) to the patients' records

* For how many patients do we have clinical information? (i.e., rows in `df_patients`)
* Do all patients have an associated location information?

In [21]:
# solution
len(df_patients)

6324

In [22]:
# solution
len(df_locations)

5

### **Exercise** : 

Combine the two information from the two tables into one, where all patient informaiton is available.

In [25]:
# solution

# Explore the date
len(df_patients['patient-id'].unique())

1324

In [26]:
# solution

len(df_patients['location-id'].unique())

5

In [27]:
# solution

1324*5

6620

Looks like not all patients have been tested at all locations.

#### Solution 1 - O(N*M)

In [59]:
# %%timeit
# solution

patients_with_city = df_patients.copy()
patients_with_city['city'] = 'n/a'

for idx, row in patients_with_city.iterrows():  # O(N)
    location = row['location-id']
    matching_city = (df_locations['location-id'] == location)   # O(M)
    city = df_locations.loc[matching_city, 'city']
    if len(city) > 0:
        patients_with_city.loc[idx, 'city'] = city.iloc[0]

#### Solution 2 - 
- with sorting

In [31]:
patients_with_group['group_2'] = 'n/a'

sorted_patients = patients_with_group.sort_values(['patient-id', 'location-id'])   # O(N log N)
sorted_groups = groups.sort_values(['patient-id', 'location-id'])                  # O(M log M)

group_2_col = sorted_patients.columns.get_loc('group_2')
groups_idx = 0
patients_idx = 0

while True:    # O(N + M)
    row_groups = sorted_groups.iloc[groups_idx]
    key_groups = (row_groups['patient-id'], row_groups['location-id'])
    
    row_patients = sorted_patients.iloc[patients_idx]
    key_patients = (row_patients['patient-id'], row_patients['location-id'])

    if key_patients == key_groups:
        original_idx = sorted_patients.index[patients_idx]
        patients_with_group.iloc[original_idx, group_2_col] = row_groups['group']
        patients_idx += 1
    elif key_patients < key_groups:
        # missing data
        patients_idx += 1
    else:
        groups_idx += 1
        if groups_idx >= len(sorted_groups):
          break
    if patients_idx >= len(sorted_patients):
        break

#### Solution - O(n+m)

In [41]:
# %%timeit
# solution - optimal
df_with_info = df_patients.merge(df_locations, on = ['patient-id', 'location-id'], how = 'left')

In [1]:
# solution
# same as above but 'by hand'

patients_with_group['group_3'] = 'n/a'
# build hash table: O(M)
group_lookup = {
  (row['patient-id'], row['location-id']): row['group']
  for _, row in groups.iterrows()
}

# probe hash table once per row: O(N)
group_col = [
  group_lookup.get((patient, location), np.nan)
  for patient, location in zip(patients['patient-id'], patients['location-id'])
]

patients_with_group['group_3'] = group_col

NameError: name 'patients_with_group' is not defined

# 4. Save final result in `processed_data_predimed.csv`

1. Using the `.to_csv` method of Pandas DataFrames

In [19]:
df_without_dropped.to_csv('processed_data_predimed.csv', index=None)